In [1]:
# Imports section
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression  
from sklearn.preprocessing import PolynomialFeatures

## Part 1. Loading the dataset

In [2]:
# Using pandas load the dataset (load remotely, not locally)
df = pd.read_csv('science_data_large.csv')
# Output the first 15 rows of the data
print(df.head(15))
# Display a summary of the table information (number of datapoints, etc.)
print(df.describe())

    Temperature °C  Mols KCL     Size nm^3
0              469       647  6.244743e+05
1              403       694  5.779610e+05
2              302       975  6.196847e+05
3              779       916  1.460449e+06
4              901        18  4.325726e+04
5              545       637  7.124634e+05
6              660       519  7.006960e+05
7              143       869  2.718260e+05
8               89       461  8.919803e+04
9              294       776  4.770210e+05
10             991       117  2.441771e+05
11             307       781  5.006455e+05
12             206        70  3.145200e+04
13             437       599  5.390215e+05
14             566        75  9.185271e+04
       Temperature °C     Mols KCL     Size nm^3
count     1000.000000  1000.000000  1.000000e+03
mean       500.500000   471.530000  5.086111e+05
std        288.819436   288.482872  4.474838e+05
min          1.000000     1.000000  1.611429e+01
25%        250.750000   226.750000  1.298267e+05
50%        500.500

## Part 2. Splitting the dataset

In [3]:
# Take the pandas dataset and split it into our features (X) and label (y)
X = df.drop(columns=['Size nm^3'])  # Features
y = df['Size nm^3']                 # Label
# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42) #Use a random state so that the X and y data still correspond with one another

print(X_test)
print(y_test)

     Temperature °C  Mols KCL
521             100       541
737             635       668
740             799       665
660             966       871
411             785       595
..              ...       ...
436             421       725
764             431       860
88              983       433
63              580       817
826              24        89

[100 rows x 2 columns]
521    1.177623e+05
737    8.687293e+05
740    1.084893e+06
660    1.716039e+06
411    9.536850e+05
           ...     
436    6.305199e+05
764    7.676234e+05
88     8.684308e+05
63     9.737511e+05
826    4.786314e+03
Name: Size nm^3, Length: 100, dtype: float64


## Part 3. Perform a Linear Regression

In [4]:
# Use sklearn to train a model on the training set
model = LinearRegression()
model.fit(X_train, y_train)

# Create a sample datapoint and predict the output of that sample with the trained model
datapoint = X_test.head(1)
y_pred = model.predict(datapoint)
print('y prediction:',y_pred[0])

# Report on the score for that model, in your own words (markdown, not code) explain what the score means
score = model.score(X_test, y_test)
print(score)
# Extract the coefficents and intercept from the model and write an equation for your h(x) using LaTeX
coefficients = model.coef_
intercept = model.intercept_
print(coefficients, intercept)

y prediction: 235911.1927226002
0.8552472077276095
[ 866.14641337 1032.69506649] -409391.47958340764


The score for this model is .855 which is the $R^2$ score. What $R^2$ shows is the proportion of variance in the label that can be explained by the independent variables. A score of 1 means all of the variance can be explained by the independent variables and a score of 0 means none of the variance can be explained by the independent variables. A score of .855 means that 85.5% of the variance in the label is predictable from the features.

$x_1$ is the temperature and $x_2$ is the moles of potassium chloride

$h(x) = 866.146 \cdot x_1 + 1032.695 \cdot x_2 -409391.47$

## Part 4. Use Cross Validation

In [5]:
# Use the cross_val_score function to repeat your experiment across many shuffles of the data
scores = cross_val_score(model, X, y, cv=5)
# Report on their finding and their significance
print(scores)
print("Mean R-squared score:", np.mean(scores))

[0.83918826 0.87051239 0.85871066 0.87202623 0.84364641]
Mean R-squared score: 0.8568167899144437


The cross validation score splits the data into 4 parts for training and 1 part for testing and this process happens 5 times. We get the $R^2$ score for each of these times and then take the mean. We get a mean score of .857 which means that 85.7% of the variance in the label is predictable from its features. 

## Part 5. Using Polynomial Regression

In [6]:
# Using the PolynomialFeatures library perform another regression on an augmented dataset of degree 2
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_poly, y, test_size=0.1, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)

# Report on the metrics and output the resultant equation as you did in Part 3.
score = model.score(X_test, y_test)
print(score)
coefficients = model.coef_
intercept = model.intercept_
print(coefficients, intercept)




1.0
[ 0.00000000e+00  1.20000000e+01 -1.27195488e-07  1.26494371e-11
  2.00000000e+00  2.85714287e-02] 2.0477105863392353e-05


a is Temperature and b is Moles of Potassium Chloride
$h(x) = 12 \cdot a + 2 \cdot ab + .02857 \cdot b^2$ 

The $R^2$ score for the polynomial regression is 1 which means this data is a perfect fit to our labels. This could mean that our model could be overfit and does not generalize well. 